# DRI-23 INT4 Quantization With LLM Compressor

This notebook replaces the AutoGPTQ path with vLLM's maintained LLM Compressor GPTQ flow. It reruns the same DRI-23 steps: checkout repo, install CUDA deps, authenticate, regenerate TBX11K JSONL, merge the run #4 LoRA adapter into Qwen2-VL, quantize the merged model to W4A16 GPTQ, evaluate against the full validation set, and upload the quantized artifact plus eval outputs to Hugging Face.

Important: the official LLM Compressor Qwen2-VL recipe quantizes the language-side linear layers and leaves the vision tower unquantized. That is the supported path for multimodal Qwen right now; the final artifact may be larger than the old <4 GB target, but it should be much less brittle than AutoGPTQ.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri23-gptq-quantization"
PROJECT_DIR = Path("/content/Drishti")

BASE_MODEL = "Qwen/Qwen2-VL-7B-Instruct"
ADAPTER_REPO_ID = "ShivSingh123/drishti-qlora-run4-vision-lora-ablation"
ADAPTER_REPO_PATH = "checkpoints/checkpoint-4950"

MERGED_DIR = Path("outputs/dri23-run4-merged-fp16")
QUANT_DIR = Path("outputs/dri23-run4-llmcompressor-gptq-int4")
SMOKE_EVAL_DIR = Path("outputs/eval/dri23-run4-llmcompressor-gptq-int4-smoke")
FULL_EVAL_DIR = Path("outputs/eval/dri23-run4-llmcompressor-gptq-int4-full-val")
HF_QUANT_REPO_ID = "ShivSingh123/drishti-qwen2vl-run4-llmcompressor-gptq-int4"

CALIBRATION_SAMPLES = 128
MAX_SEQUENCE_LENGTH = 2048

In [ ]:
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run(command: list[str], *, cwd: Path | None = None, capture: bool = False) -> subprocess.CompletedProcess | None:
    print("
$ " + " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        capture_output=capture,
    )
    if capture:
        print(result.stdout)
        print(result.stderr)
    result.check_returncode()
    return result if capture else None

In [ ]:
if not PROJECT_DIR.exists():
    run(["git", "clone", PROJECT_REPO, str(PROJECT_DIR)])

run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)

%cd /content/Drishti
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "llmcompressor[qwen]"])

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
    kaggle_username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
else:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    kaggle_username = os.environ.get("KAGGLE_USERNAME")
    kaggle_key = os.environ.get("KAGGLE_KEY")

if not hf_token:
    raise RuntimeError("Add HF_TOKEN or HUGGINGFACE_TOKEN to Colab secrets before continuing.")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

if kaggle_username and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
else:
    print("Kaggle secrets not found in Colab. download_dataset.py may still work if kaggle.json already exists.")

api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_QUANT_REPO_ID, repo_type="model", private=True, exist_ok=True)
print(f"HF repo ready: {HF_QUANT_REPO_ID}")

In [ ]:
!nvidia-smi

import importlib.metadata as md
import torch

for package in ["torch", "transformers", "peft", "accelerate", "datasets", "qwen-vl-utils", "llmcompressor", "compressed-tensors"]:
    try:
        print(package, md.version(package))
    except Exception as exc:
        print(package, "missing", exc)

print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
run(["python", "download_dataset.py"])
run(["python", "generate_jsonl.py", "--output-dir", "data/processed"])

import json
from collections import Counter

expected = {
    "train": Counter({"active_tb": 600, "healthy": 3000, "sick_but_non_tb": 3000}),
    "val": Counter({"active_tb": 200, "healthy": 800, "sick_but_non_tb": 800}),
}

for split, expected_counts in expected.items():
    counts = Counter()
    with open(f"data/processed/{split}.jsonl", encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            assistant = payload["messages"][1]["content"]
            assert assistant.startswith("Classification: "), assistant
            assert assistant.count("
") == 0, assistant
            counts[assistant.removeprefix("Classification: ")] += 1
    print(split, dict(sorted(counts.items())))
    assert counts == expected_counts, (split, counts, expected_counts)

## Merge Run #4 LoRA Into Qwen2-VL

This recreates the merged full-precision checkpoint locally. If the directory already exists from a previous notebook attempt, the cell skips the merge.

In [ ]:
if MERGED_DIR.exists():
    print(f"Skipping merge because {MERGED_DIR} already exists.")
else:
    run([
        "python", "merge_lora_checkpoint.py",
        "--base-model", BASE_MODEL,
        "--adapter-repo-id", ADAPTER_REPO_ID,
        "--adapter-repo-path", ADAPTER_REPO_PATH,
        "--output-dir", str(MERGED_DIR),
        "--torch-dtype", "bfloat16",
    ])

## Quantize With LLM Compressor GPTQ

This uses W4A16 GPTQ with 128 validation calibration samples. It follows the official Qwen2-VL multimodal recipe: quantize `Linear` layers, ignore `lm_head`, and ignore the vision tower with `re:visual.*` / `re:model.visual.*`.

In [ ]:
run([
    "python", "quantize_llmcompressor_qwen2vl.py",
    "--merged-model-dir", str(MERGED_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--calibration-samples", str(CALIBRATION_SAMPLES),
    "--output-dir", str(QUANT_DIR),
    "--scheme", "W4A16",
    "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
    "--debug-traceback",
])

In [ ]:
import json
metadata_path = QUANT_DIR / "quantization_metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2))

## Smoke Evaluate

This runs a tiny stratified eval first. If this fails to load the compressed model or the locked output smoke fails, stop here and send me the traceback.

In [ ]:
run([
    "python", "evaluate_quantized_checkpoint.py",
    "--model-dir", str(QUANT_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--output-dir", str(SMOKE_EVAL_DIR),
    "--limit-per-class", "5",
    "--generation-smoke",
    "--gate", "run3-full",
    "--max-macro-f1-drop", "0.02",
])

## Full Validation Eval

This is the real acceptance check against the 1,800-sample validation set. It fails if macro-F1 drops by more than 0.02 from the run #4 full-precision baseline.

In [ ]:
run([
    "python", "evaluate_quantized_checkpoint.py",
    "--model-dir", str(QUANT_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--output-dir", str(FULL_EVAL_DIR),
    "--batch-size", "3",
    "--generation-smoke",
    "--gate", "run3-full",
    "--max-macro-f1-drop", "0.02",
    "--fail-on-gate-fail",
])

In [ ]:
import json
metrics = json.loads((FULL_EVAL_DIR / "eval_results.json").read_text(encoding="utf-8"))
print(f"accuracy: {metrics['accuracy']:.6f}")
print(f"macro_f1: {metrics['macro_f1']:.6f}")
print(f"macro_f1_delta: {metrics['quantization_delta']['macro_f1_delta']:.6f}")
print(f"macro_f1_drop: {metrics['quantization_delta']['macro_f1_drop']:.6f}")
print(f"delta_gate_passed: {metrics['quantization_delta']['passed']}")
print(f"structured_output_passed: {metrics.get('structured_output_smoke', {}).get('passed')}")
print("prediction_distribution:", metrics["prediction_distribution"])
for label, values in metrics["per_class"].items():
    print(label, values)

## Upload Quantized Model And Eval Artifacts

Run this only after the smoke and full validation cells look good.

In [ ]:
api.upload_folder(
    repo_id=HF_QUANT_REPO_ID,
    repo_type="model",
    folder_path=str(QUANT_DIR),
    path_in_repo=".",
    commit_message="Upload run4 LLM Compressor W4A16 GPTQ checkpoint",
)

api.upload_folder(
    repo_id=HF_QUANT_REPO_ID,
    repo_type="model",
    folder_path=str(FULL_EVAL_DIR),
    path_in_repo="eval/full-val",
    commit_message="Upload full validation eval for LLM Compressor GPTQ checkpoint",
)
print(f"Uploaded to https://huggingface.co/{HF_QUANT_REPO_ID}")